In [1]:
# import os
# os.environ["OPENAI_API_KEY"] = "sk-proj-"
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

In [3]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [4]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [5]:
vector_store = Chroma(
    embedding_function=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", dimensions=32),
    persist_directory='my_chroma_db',
    collection_name='sample'
)

In [6]:
# add documents
vector_store.add_documents(docs)

['ae90435d-9164-4528-83d4-b7490c91781d',
 'd6ec5ad1-13b0-45d8-9cd6-49044b004f33',
 'a310a5a9-2312-4432-9cc9-ad5565782b4c',
 'd7a8f171-1089-43b6-b450-1027d7dda378',
 'b07ff774-82bb-48d4-9deb-ff5b3159d9bf']

In [7]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['ae90435d-9164-4528-83d4-b7490c91781d',
  'd6ec5ad1-13b0-45d8-9cd6-49044b004f33',
  'a310a5a9-2312-4432-9cc9-ad5565782b4c',
  'd7a8f171-1089-43b6-b450-1027d7dda378',
  'b07ff774-82bb-48d4-9deb-ff5b3159d9bf'],
 'embeddings': array([[ 0.0074764 ,  0.01034819,  0.01265225, ...,  0.00313006,
          0.01334844,  0.00201392],
        [ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, -0.01345706, ...,  0.00715038,
          0.01247527,  0.00910379]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

In [14]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='d7a8f171-1089-43b6-b450-1027d7dda378', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='b07ff774-82bb-48d4-9deb-ff5b3159d9bf', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [21]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="who among these is an all rounder?",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='b07ff774-82bb-48d4-9deb-ff5b3159d9bf', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.5834747552871704),
 (Document(id='a310a5a9-2312-4432-9cc9-ad5565782b4c', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.7325684428215027)]

In [22]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='d7a8f171-1089-43b6-b450-1027d7dda378', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6193535327911377),
 (Document(id='b07ff774-82bb-48d4-9deb-ff5b3159d9bf', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6677330136299133)]

In [23]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

In [24]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['ae90435d-9164-4528-83d4-b7490c91781d',
  'd6ec5ad1-13b0-45d8-9cd6-49044b004f33',
  'a310a5a9-2312-4432-9cc9-ad5565782b4c',
  'd7a8f171-1089-43b6-b450-1027d7dda378',
  'b07ff774-82bb-48d4-9deb-ff5b3159d9bf'],
 'embeddings': array([[ 0.0074764 ,  0.01034819,  0.01265225, ...,  0.00313006,
          0.01334844,  0.00201392],
        [ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, -0.01345706, ...,  0.00715038,
          0.01247527,  0.00910379]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

In [25]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [26]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['ae90435d-9164-4528-83d4-b7490c91781d',
  'd6ec5ad1-13b0-45d8-9cd6-49044b004f33',
  'a310a5a9-2312-4432-9cc9-ad5565782b4c',
  'd7a8f171-1089-43b6-b450-1027d7dda378',
  'b07ff774-82bb-48d4-9deb-ff5b3159d9bf'],
 'embeddings': array([[ 0.0074764 ,  0.01034819,  0.01265225, ...,  0.00313006,
          0.01334844,  0.00201392],
        [ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, -0.01345706, ...,  0.00715038,
          0.01247527,  0.00910379]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m